# Pré-processamento 8 — Revisão de compatibilidade entre palavra e categoria
## Decisões aprovadas
Aplicar 35 correções do grupo Nome Comun Masculino Singular e 24 do grupo Nome Comun Feminino Singular e 82 do grupo Nome Própio Pessoa e 11 do grupo Verbo Principal Infinitivo e 28 do grupo Verbo Principal Indicativo Presente Terceira Pessoa Singular (180 aprovadas ao todo). Não alterar LEMA.
O código aceita categorias já corrigidas, permitindo reexecução. Valida palavras pela identidade e pelo lema, não pela antiga posição da linha. As linhas históricas abaixo se referem à versão anterior.
As análises linguísticas são revisão humana assistida; o script reproduz decisões explícitas, não um classificador automático.
Para acrescentar remoções, edite a lista e reexecute desde o carregamento.
O salvamento mostra as contagens da execução e o total final da base. Arquivos intermediários são criados em pasta temporária e limpos automaticamente ao concluir ou ocorrer uma exceção. As versões anteriores são mantidas pelo Git.

In [15]:
from pathlib import Path
import json, hashlib, os, subprocess
import pandas as pd
from IPython.display import display
arquivo = Path("Corpus_SANTOS_TONI_FDC_tratado_sem_repeticoes.xlsx")
base = pd.read_excel(arquivo)
base_original = base.copy(deep=True)
hash_entrada = hashlib.sha256(arquivo.read_bytes()).hexdigest()
print("Registros carregados:", len(base))


Registros carregados: 7785


## 1. Função de correção de categoria e preparação do resultado

In [16]:
# Função compartilhada pelos grupos; todo o código de correção está neste notebook.
# Altera exclusivamente CATMORF, sem atribuir valores em LEMA.
def aplicar_correcoes_categoria(resultado, correcoes):
    antes = resultado.copy(deep=True)
    aplicadas = []
    for item in correcoes:
        mascara = resultado["PALAVRA"].eq(item["PALAVRA"])
        assert mascara.sum() == 1, f"Palavra ausente ou duplicada: {item['PALAVRA']}"
        indice = resultado.index[mascara][0]
        atual = resultado.at[indice, "CATMORF"]
        assert atual in (item["CATMORF"], item["CATEGORIA_SUGERIDA"]), f"Categoria mudou: {item['PALAVRA']}"
        assert resultado.at[indice, "LEMA"] == item["LEMA"], f"Lema mudou: {item['PALAVRA']}"
        if atual != item["CATEGORIA_SUGERIDA"]:
            resultado.at[indice, "CATMORF"] = item["CATEGORIA_SUGERIDA"]
            aplicadas.append({**item, "CATEGORIA_ANTERIOR": atual})
    pd.testing.assert_frame_equal(antes.drop(columns="CATMORF"), resultado.drop(columns="CATMORF"))
    return aplicadas

resultado = base.copy(deep=True)
alteracoes = []
correcoes_aprovadas = []


## 1.1. Nome Comun Masculino Singular — aplicar 35 correções aprovadas

In [3]:
# Lista completa das decisões aprovadas — Nome Comun Masculino Singular
correcoes_masculino = [{'LINHA_EXCEL': 237,
  'PALAVRA': 'ajudando',
  'LEMA': 'ajudá-lo',
  'CATMORF': 'Nome Comun Masculino Singular',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Gerúndio',
  'JUSTIFICATIVA': 'Gerúndio de ajudar; não corresponde a substantivo masculino singular.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 246,
  'PALAVRA': 'ajudá-lo',
  'LEMA': 'ajudador',
  'CATMORF': 'Nome Comun Masculino Singular',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Infinitivo',
  'JUSTIFICATIVA': 'Infinitivo verbal com pronome enclítico; classificar o componente verbal, '
                   'preservando a palavra completa.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 253,
  'PALAVRA': 'albuns',
  'LEMA': 'alazão',
  'CATMORF': 'Nome Comun Masculino Singular',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Plural',
  'JUSTIFICATIVA': 'Forma plural de álbum, sem acento no dado original; propõe-se apenas corrigir '
                   'o número, preservando a grafia.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 258,
  'PALAVRA': 'alcaparra',
  'LEMA': 'alcanço',
  'CATMORF': 'Nome Comun Masculino Singular',
  'CATEGORIA_SUGERIDA': 'Nome Comun Feminino Singular',
  'JUSTIFICATIVA': 'Substantivo de gênero feminino; a classificação masculina não corresponde à '
                   'forma.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 261,
  'PALAVRA': 'aldeia',
  'LEMA': 'alce',
  'CATMORF': 'Nome Comun Masculino Singular',
  'CATEGORIA_SUGERIDA': 'Nome Comun Feminino Singular',
  'JUSTIFICATIVA': 'Substantivo de gênero feminino; a classificação masculina não corresponde à '
                   'forma.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 269,
  'PALAVRA': 'alergia',
  'LEMA': 'alemão',
  'CATMORF': 'Nome Comun Masculino Singular',
  'CATEGORIA_SUGERIDA': 'Nome Comun Feminino Singular',
  'JUSTIFICATIVA': 'Substantivo de gênero feminino; a classificação masculina não corresponde à '
                   'forma.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 274,
  'PALAVRA': 'algo',
  'LEMA': 'alfinete',
  'CATMORF': 'Nome Comun Masculino Singular',
  'CATEGORIA_SUGERIDA': 'Pronome Indefinido Invariável Indefinido',
  'JUSTIFICATIVA': 'Indefinido invariável, também empregado como advérbio; a proposta pronominal é '
                   'uma análise possível.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 276,
  'PALAVRA': 'algum',
  'LEMA': 'algodão',
  'CATMORF': 'Nome Comun Masculino Singular',
  'CATEGORIA_SUGERIDA': 'Determinante Indefinido Masculino Singular Indefinido',
  'JUSTIFICATIVA': 'Indefinido masculino singular; pode funcionar como determinante ou pronome, '
                   'não como nome comum nessa análise lexical.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 291,
  'PALAVRA': 'almofadas',
  'LEMA': 'almoço',
  'CATMORF': 'Nome Comun Masculino Singular',
  'CATEGORIA_SUGERIDA': 'Nome Comun Feminino Plural',
  'JUSTIFICATIVA': 'Substantivo feminino no plural; gênero e número incompatíveis com masculino '
                   'singular.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 292,
  'PALAVRA': 'almoça',
  'LEMA': 'alívio',
  'CATMORF': 'Nome Comun Masculino Singular',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Indicativo Presente Terceira Pessoa Singular',
  'JUSTIFICATIVA': 'Forma de almoçar; também admite imperativo. Presente do indicativo é uma opção '
                   'válida, sem preferência entre análises verbais possíveis.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 304,
  'PALAVRA': 'aluga',
  'LEMA': 'alto',
  'CATMORF': 'Nome Comun Masculino Singular',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Indicativo Presente Terceira Pessoa Singular',
  'JUSTIFICATIVA': 'Forma de alugar; também admite imperativo. A proposta é uma análise verbal '
                   'possível.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 309,
  'PALAVRA': 'alunos',
  'LEMA': 'aluguel',
  'CATMORF': 'Nome Comun Masculino Singular',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Plural',
  'JUSTIFICATIVA': 'Forma plural do substantivo; a classificação singular não corresponde à '
                   'palavra.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 351,
  'PALAVRA': 'ameaçou',
  'LEMA': 'ambiente',
  'CATMORF': 'Nome Comun Masculino Singular',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Indicativo Passado Terceira Pessoa Singular',
  'JUSTIFICATIVA': 'Forma de ameaçar no pretérito perfeito, terceira pessoa do singular.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 357,
  'PALAVRA': 'americanas',
  'LEMA': 'amendoim',
  'CATMORF': 'Nome Comun Masculino Singular',
  'CATEGORIA_SUGERIDA': 'Nome Comun Feminino Plural',
  'JUSTIFICATIVA': 'Admite leitura como substantivo, mas na forma feminina plural; preservar a '
                   'classe nominal e corrigir gênero/número.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 361,
  'PALAVRA': 'amigas',
  'LEMA': 'americano',
  'CATMORF': 'Nome Comun Masculino Singular',
  'CATEGORIA_SUGERIDA': 'Nome Comun Feminino Plural',
  'JUSTIFICATIVA': 'Admite leitura como substantivo, mas na forma feminina plural; preservar a '
                   'classe nominal e corrigir gênero/número.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 364,
  'PALAVRA': 'amizade',
  'LEMA': 'amigo',
  'CATMORF': 'Nome Comun Masculino Singular',
  'CATEGORIA_SUGERIDA': 'Nome Comun Feminino Singular',
  'JUSTIFICATIVA': 'Substantivo de gênero feminino; a classificação masculina não corresponde à '
                   'forma.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 369,
  'PALAVRA': 'amostra',
  'LEMA': 'amo',
  'CATMORF': 'Nome Comun Masculino Singular',
  'CATEGORIA_SUGERIDA': 'Nome Comun Feminino Singular',
  'JUSTIFICATIVA': 'Substantivo de gênero feminino; a classificação masculina não corresponde à '
                   'forma.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 370,
  'PALAVRA': 'amostras',
  'LEMA': 'amor',
  'CATMORF': 'Nome Comun Masculino Singular',
  'CATEGORIA_SUGERIDA': 'Nome Comun Feminino Plural',
  'JUSTIFICATIVA': 'Substantivo feminino no plural; gênero e número incompatíveis com masculino '
                   'singular.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 379,
  'PALAVRA': 'anda',
  'LEMA': 'anão',
  'CATMORF': 'Nome Comun Masculino Singular',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Indicativo Presente Terceira Pessoa Singular',
  'JUSTIFICATIVA': 'Forma de andar ou, como substantivo, feminino (anda/perna de pau). Masculino '
                   'singular não se sustenta nessas leituras; a proposta verbal é possível.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 402,
  'PALAVRA': 'anoitecendo',
  'LEMA': 'anjo',
  'CATMORF': 'Nome Comun Masculino Singular',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Gerúndio',
  'JUSTIFICATIVA': 'Gerúndio de anoitecer; não corresponde a substantivo masculino singular.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 403,
  'PALAVRA': 'anos',
  'LEMA': 'ano',
  'CATMORF': 'Nome Comun Masculino Singular',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Plural',
  'JUSTIFICATIVA': 'Forma plural do substantivo; a classificação singular não corresponde à '
                   'palavra.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 416,
  'PALAVRA': 'antigas',
  'LEMA': 'antídoto',
  'CATMORF': 'Nome Comun Masculino Singular',
  'CATEGORIA_SUGERIDA': 'Adjetivo Qualificativo Feminino Plural',
  'JUSTIFICATIVA': 'Forma feminina plural de antigo. Mesmo em leitura substantivada, masculino '
                   'singular é incompatível; adjetivo é uma proposta possível.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 2247,
  'PALAVRA': 'der',
  'LEMA': 'der',
  'CATMORF': 'Nome Comun Masculino Singular',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Subjuntivo Futuro Terceira Pessoa Singular',
  'JUSTIFICATIVA': 'Forma do futuro do subjuntivo de dar; primeira e terceira pessoas do singular '
                   'são possíveis. Propõe-se uma delas sem preferência.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 2251,
  'PALAVRA': 'derrete',
  'LEMA': 'derrete',
  'CATMORF': 'Nome Comun Masculino Singular',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Indicativo Presente Terceira Pessoa Singular',
  'JUSTIFICATIVA': 'Forma de derreter; também admite imperativo. A proposta é uma análise verbal '
                   'possível.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 2463,
  'PALAVRA': 'dica',
  'LEMA': 'dica',
  'CATMORF': 'Nome Comun Masculino Singular',
  'CATEGORIA_SUGERIDA': 'Nome Comun Feminino Singular',
  'JUSTIFICATIVA': 'Substantivo de gênero feminino; a classificação masculina não corresponde à '
                   'forma.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 2483,
  'PALAVRA': 'dinamite',
  'LEMA': 'dinamite',
  'CATMORF': 'Nome Comun Masculino Singular',
  'CATEGORIA_SUGERIDA': 'Nome Comun Feminino Singular',
  'JUSTIFICATIVA': 'Substantivo de gênero feminino; a classificação masculina não corresponde à '
                   'forma.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 2538,
  'PALAVRA': 'devagarzinho',
  'LEMA': 'devagarzinho',
  'CATMORF': 'Nome Comun Masculino Singular',
  'CATEGORIA_SUGERIDA': 'Advérbio Geral',
  'JUSTIFICATIVA': 'Forma diminutiva de devagar, com valor adverbial de modo.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 2662,
  'PALAVRA': 'dói',
  'LEMA': 'dói',
  'CATMORF': 'Nome Comun Masculino Singular',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Indicativo Presente Terceira Pessoa Singular',
  'JUSTIFICATIVA': 'Forma de doer no presente, terceira pessoa do singular.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 2759,
  'PALAVRA': 'encaixo',
  'LEMA': 'encaixo',
  'CATMORF': 'Nome Comun Masculino Singular',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Indicativo Presente Primeira Pessoa Singular',
  'JUSTIFICATIVA': 'Forma de encaixar no presente, primeira pessoa do singular; o substantivo '
                   'usual é encaixe.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 2831,
  'PALAVRA': 'enganá-lo',
  'LEMA': 'enganá-lo',
  'CATMORF': 'Nome Comun Masculino Singular',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Infinitivo',
  'JUSTIFICATIVA': 'Infinitivo verbal com pronome enclítico; classificar o componente verbal, '
                   'preservando a palavra completa.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 4245,
  'PALAVRA': 'jogue',
  'LEMA': 'jogue',
  'CATMORF': 'Nome Comun Masculino Singular',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Subjuntivo Presente Terceira Pessoa Singular',
  'JUSTIFICATIVA': 'Forma de jogar no presente do subjuntivo ou imperativo; propõe-se uma análise '
                   'verbal possível.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 4318,
  'PALAVRA': 'lanchonete',
  'LEMA': 'lanchonete',
  'CATMORF': 'Nome Comun Masculino Singular',
  'CATEGORIA_SUGERIDA': 'Nome Comun Feminino Singular',
  'JUSTIFICATIVA': 'Substantivo de gênero feminino; a classificação masculina não corresponde à '
                   'forma.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 4408,
  'PALAVRA': 'levanto',
  'LEMA': 'levanto',
  'CATMORF': 'Nome Comun Masculino Singular',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Indicativo Presente Primeira Pessoa Singular',
  'JUSTIFICATIVA': 'Forma de levantar no presente, primeira pessoa do singular.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 6667,
  'PALAVRA': 'salvá-lo',
  'LEMA': 'salvá-lo',
  'CATMORF': 'Nome Comun Masculino Singular',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Infinitivo',
  'JUSTIFICATIVA': 'Infinitivo verbal com pronome enclítico; classificar o componente verbal, '
                   'preservando a palavra completa.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 7421,
  'PALAVRA': 'troque',
  'LEMA': 'troque',
  'CATMORF': 'Nome Comun Masculino Singular',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Subjuntivo Presente Terceira Pessoa Singular',
  'JUSTIFICATIVA': 'Forma de trocar no presente do subjuntivo ou imperativo; propõe-se uma análise '
                   'verbal possível.',
  'DECISAO': 'Aprovada'}]
# Aplicação efetiva na base em memória:
alteracoes_grupo = aplicar_correcoes_categoria(resultado, correcoes_masculino)
alteracoes.extend(alteracoes_grupo)
correcoes_aprovadas.extend(correcoes_masculino)
print("Grupo: Nome Comun Masculino Singular")
print("Correções aprovadas:", len(correcoes_masculino))
print("Aplicadas nesta execução:", len(alteracoes_grupo))
print("Já estavam corrigidas:", len(correcoes_masculino) - len(alteracoes_grupo))
display(pd.DataFrame(correcoes_masculino)[["PALAVRA", "LEMA", "CATMORF", "CATEGORIA_SUGERIDA", "DECISAO"]])


Grupo: Nome Comun Masculino Singular
Correções aprovadas: 35
Aplicadas nesta execução: 0
Já estavam corrigidas: 35


PALAVRA,LEMA,CATMORF,CATEGORIA_SUGERIDA,DECISAO
ajudando,ajudá-lo,Nome Comun Masculino Singular,Verbo Principal Gerúndio,Aprovada
ajudá-lo,ajudador,Nome Comun Masculino Singular,Verbo Principal Infinitivo,Aprovada
albuns,alazão,Nome Comun Masculino Singular,Nome Comun Masculino Plural,Aprovada
alcaparra,alcanço,Nome Comun Masculino Singular,Nome Comun Feminino Singular,Aprovada
aldeia,alce,Nome Comun Masculino Singular,Nome Comun Feminino Singular,Aprovada
alergia,alemão,Nome Comun Masculino Singular,Nome Comun Feminino Singular,Aprovada
algo,alfinete,Nome Comun Masculino Singular,Pronome Indefinido Invariável Indefinido,Aprovada
algum,algodão,Nome Comun Masculino Singular,Determinante Indefinido Masculino Singular Indefinido,Aprovada
almofadas,almoço,Nome Comun Masculino Singular,Nome Comun Feminino Plural,Aprovada
almoça,alívio,Nome Comun Masculino Singular,Verbo Principal Indicativo Presente Terceira Pessoa Singular,Aprovada


## 1.2. Nome Comun Feminino Singular — aplicar 24 correções aprovadas

In [4]:
# Lista completa das decisões aprovadas — Nome Comun Feminino Singular
correcoes_feminino = [{'LINHA_EXCEL': 235,
  'PALAVRA': 'ajudador',
  'LEMA': 'ajuda',
  'CATMORF': 'Nome Comun Feminino Singular',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Singular',
  'JUSTIFICATIVA': 'A forma ajudador é masculina; ajudadora é a correspondente feminina. Manter a '
                   'leitura nominal e corrigir o gênero.',
  'REFERENCIA': '',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 247,
  'PALAVRA': 'ajudá-los',
  'LEMA': 'ajudadora',
  'CATMORF': 'Nome Comun Feminino Singular',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Infinitivo',
  'JUSTIFICATIVA': 'Infinitivo de ajudar com pronome enclítico; não corresponde a nome comum '
                   'feminino.',
  'REFERENCIA': '',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 259,
  'PALAVRA': 'alce',
  'LEMA': 'alcaparra',
  'CATMORF': 'Nome Comun Feminino Singular',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Singular',
  'JUSTIFICATIVA': 'O substantivo que designa o animal é masculino; também há leitura verbal, mas '
                   'não é necessário preferi-la.',
  'REFERENCIA': 'https://dicionario.priberam.org/alce',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 262,
  'PALAVRA': 'alegrar',
  'LEMA': 'aldeia',
  'CATMORF': 'Nome Comun Feminino Singular',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Infinitivo',
  'JUSTIFICATIVA': 'Forma infinitiva verbal. Uma eventual leitura nominal do infinitivo seria '
                   'masculina; a categoria feminina atual não é compatível.',
  'REFERENCIA': '',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 271,
  'PALAVRA': 'alfabeto',
  'LEMA': 'alergia',
  'CATMORF': 'Nome Comun Feminino Singular',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Singular',
  'JUSTIFICATIVA': 'Substantivo masculino singular; feminino singular não corresponde à forma '
                   'lexical.',
  'REFERENCIA': '',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 294,
  'PALAVRA': 'almoçar',
  'LEMA': 'almeida',
  'CATMORF': 'Nome Comun Feminino Singular',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Infinitivo',
  'JUSTIFICATIVA': 'Forma infinitiva verbal. Uma eventual leitura nominal do infinitivo seria '
                   'masculina; a categoria feminina atual não é compatível.',
  'REFERENCIA': '',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 306,
  'PALAVRA': 'aluguel',
  'LEMA': 'altura',
  'CATMORF': 'Nome Comun Feminino Singular',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Singular',
  'JUSTIFICATIVA': 'Substantivo masculino singular; feminino singular não corresponde à forma '
                   'lexical.',
  'REFERENCIA': '',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 313,
  'PALAVRA': 'além',
  'LEMA': 'alegria',
  'CATMORF': 'Nome Comun Feminino Singular',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Singular',
  'JUSTIFICATIVA': 'Além admite substantivo masculino (o além) e advérbio; não há motivo para '
                   'preferir uma classe, então manter a nominal com gênero compatível.',
  'REFERENCIA': 'https://dicionario.priberam.org/além',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 333,
  'PALAVRA': 'amarrado',
  'LEMA': 'amarra',
  'CATMORF': 'Nome Comun Feminino Singular',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Singular',
  'JUSTIFICATIVA': 'Admite substantivo masculino (volume ou conjunto atado), além de '
                   'adjetivo/particípio. Preservar a leitura nominal possível e corrigir o gênero.',
  'REFERENCIA': 'https://dicionario.priberam.org/amarrado',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 334,
  'PALAVRA': 'amarrar',
  'LEMA': 'amarra',
  'CATMORF': 'Nome Comun Feminino Singular',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Infinitivo',
  'JUSTIFICATIVA': 'Forma infinitiva verbal. Uma eventual leitura nominal do infinitivo seria '
                   'masculina; a categoria feminina atual não é compatível.',
  'REFERENCIA': '',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 356,
  'PALAVRA': 'amendoins',
  'LEMA': 'amelinha',
  'CATMORF': 'Nome Comun Feminino Singular',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Plural',
  'JUSTIFICATIVA': 'Plural masculino de amendoim; a categoria atual erra gênero e número.',
  'REFERENCIA': '',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 362,
  'PALAVRA': 'amigo',
  'LEMA': 'amígdala',
  'CATMORF': 'Nome Comun Feminino Singular',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Singular',
  'JUSTIFICATIVA': 'Substantivo masculino singular; feminino singular não corresponde à forma '
                   'lexical.',
  'REFERENCIA': '',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 367,
  'PALAVRA': 'amor',
  'LEMA': 'amizade',
  'CATMORF': 'Nome Comun Feminino Singular',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Singular',
  'JUSTIFICATIVA': 'Substantivo masculino singular; feminino singular não corresponde à forma '
                   'lexical.',
  'REFERENCIA': '',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 376,
  'PALAVRA': 'amígdalas',
  'LEMA': 'amigo',
  'CATMORF': 'Nome Comun Feminino Singular',
  'CATEGORIA_SUGERIDA': 'Nome Comun Feminino Plural',
  'JUSTIFICATIVA': 'Plural de amígdala; corrigir apenas o número.',
  'REFERENCIA': '',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 380,
  'PALAVRA': 'andando',
  'LEMA': 'âncora',
  'CATMORF': 'Nome Comun Feminino Singular',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Gerúndio',
  'JUSTIFICATIVA': 'Gerúndio de andar; a categoria de nome comum feminino singular não corresponde '
                   'à forma.',
  'REFERENCIA': '',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 413,
  'PALAVRA': 'antes',
  'LEMA': 'antena',
  'CATMORF': 'Nome Comun Feminino Singular',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Singular',
  'JUSTIFICATIVA': 'Antes admite substantivo masculino (o antes e o depois), além de advérbio; '
                   'manter a leitura nominal com gênero compatível.',
  'REFERENCIA': 'https://dicionario.priberam.org/antes',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 1662,
  'PALAVRA': 'cocoricó',
  'LEMA': 'cocoricó',
  'CATMORF': 'Nome Comun Feminino Singular',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Singular',
  'JUSTIFICATIVA': 'A onomatopeia admite leitura nominal masculina para o canto do galo; preservar '
                   'essa classe e corrigir o gênero.',
  'REFERENCIA': 'https://dicionario.priberam.org/cocoricó',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 2452,
  'PALAVRA': 'dezessete',
  'LEMA': 'dezessete',
  'CATMORF': 'Nome Comun Feminino Singular',
  'CATEGORIA_SUGERIDA': 'Quantidade',
  'JUSTIFICATIVA': 'Numeral cardinal; Quantidade é a classificação usada na base para numerais. '
                   'Também admite uso nominal masculino, mas feminino singular não corresponde à '
                   'leitura lexical.',
  'REFERENCIA': '',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 2625,
  'PALAVRA': 'drible',
  'LEMA': 'drible',
  'CATMORF': 'Nome Comun Feminino Singular',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Singular',
  'JUSTIFICATIVA': 'O substantivo drible é masculino; também há leitura verbal, que não precisa '
                   'ser preferida.',
  'REFERENCIA': 'https://dicionario.priberam.org/drible',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 3644,
  'PALAVRA': 'fubá',
  'LEMA': 'fubá',
  'CATMORF': 'Nome Comun Feminino Singular',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Singular',
  'JUSTIFICATIVA': 'Na leitura nominal, fubá é masculino.',
  'REFERENCIA': 'https://dicionario.priberam.org/fubá',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 3885,
  'PALAVRA': 'guarda-chuva',
  'LEMA': 'guarda-chuva',
  'CATMORF': 'Nome Comun Feminino Singular',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Singular',
  'JUSTIFICATIVA': 'O composto é substantivo masculino; o gênero de chuva não determina o gênero '
                   'do composto.',
  'REFERENCIA': 'https://dicionario.priberam.org/guarda-chuva',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 6210,
  'PALAVRA': 'quebra-cabeça',
  'LEMA': 'quebra-cabeça',
  'CATMORF': 'Nome Comun Feminino Singular',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Singular',
  'JUSTIFICATIVA': 'O composto é substantivo masculino; não acompanha o gênero de cabeça.',
  'REFERENCIA': 'https://dicionario.priberam.org/quebra-cabeça',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 6653,
  'PALAVRA': 'samba',
  'LEMA': 'samba',
  'CATMORF': 'Nome Comun Feminino Singular',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Singular',
  'JUSTIFICATIVA': 'Na leitura nominal registrada para dança, música ou baile, samba é masculino; '
                   'preservar a classe nominal.',
  'REFERENCIA': 'https://dicionario.priberam.org/samba',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 7796,
  'PALAVRA': 'ímã',
  'LEMA': 'ímã',
  'CATMORF': 'Nome Comun Feminino Singular',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Singular',
  'JUSTIFICATIVA': 'O substantivo que designa o objeto magnético é masculino.',
  'REFERENCIA': 'https://dicionario.priberam.org/ímã',
  'DECISAO': 'Aprovada'}]
# Aplicação efetiva na base em memória:
alteracoes_grupo = aplicar_correcoes_categoria(resultado, correcoes_feminino)
alteracoes.extend(alteracoes_grupo)
correcoes_aprovadas.extend(correcoes_feminino)
print("Grupo: Nome Comun Feminino Singular")
print("Correções aprovadas:", len(correcoes_feminino))
print("Aplicadas nesta execução:", len(alteracoes_grupo))
print("Já estavam corrigidas:", len(correcoes_feminino) - len(alteracoes_grupo))
display(pd.DataFrame(correcoes_feminino)[["PALAVRA", "LEMA", "CATMORF", "CATEGORIA_SUGERIDA", "DECISAO"]])


Grupo: Nome Comun Feminino Singular
Correções aprovadas: 24
Aplicadas nesta execução: 0
Já estavam corrigidas: 24


PALAVRA,LEMA,CATMORF,CATEGORIA_SUGERIDA,DECISAO
ajudador,ajuda,Nome Comun Feminino Singular,Nome Comun Masculino Singular,Aprovada
ajudá-los,ajudadora,Nome Comun Feminino Singular,Verbo Principal Infinitivo,Aprovada
alce,alcaparra,Nome Comun Feminino Singular,Nome Comun Masculino Singular,Aprovada
alegrar,aldeia,Nome Comun Feminino Singular,Verbo Principal Infinitivo,Aprovada
alfabeto,alergia,Nome Comun Feminino Singular,Nome Comun Masculino Singular,Aprovada
almoçar,almeida,Nome Comun Feminino Singular,Verbo Principal Infinitivo,Aprovada
aluguel,altura,Nome Comun Feminino Singular,Nome Comun Masculino Singular,Aprovada
além,alegria,Nome Comun Feminino Singular,Nome Comun Masculino Singular,Aprovada
amarrado,amarra,Nome Comun Feminino Singular,Nome Comun Masculino Singular,Aprovada
amarrar,amarra,Nome Comun Feminino Singular,Verbo Principal Infinitivo,Aprovada


## 1.3. Nome Própio Pessoa — aplicar 82 correções aprovadas


In [5]:
# Lista completa das decisões aprovadas — Nome Própio Pessoa
correcoes_pessoas = [{'LINHA_EXCEL': 14,
  'PALAVRA': 'abdômen',
  'LEMA': 'abdômen',
  'CATMORF': 'Nome Própio Pessoa',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Singular',
  'JUSTIFICATIVA': 'Na leitura lexical, é nome comum masculino singular. Se a ocorrência for '
                   'apelido ou nome de personagem, manter a categoria atual.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 243,
  'PALAVRA': 'ajudei',
  'LEMA': 'ajudaro',
  'CATMORF': 'Nome Própio Pessoa',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Indicativo Passado Primeira Pessoa Singular',
  'JUSTIFICATIVA': 'Pretérito perfeito de ajudar, primeira pessoa do singular.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 283,
  'PALAVRA': 'alimentar',
  'LEMA': 'alice',
  'CATMORF': 'Nome Própio Pessoa',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Infinitivo',
  'JUSTIFICATIVA': 'Admite leitura de infinitivo; matá-la inclui pronome enclítico. Alimentar '
                   'também pode ser adjetivo. A proposta não estabelece preferência entre análises '
                   'possíveis.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 293,
  'PALAVRA': 'almoçando',
  'LEMA': 'all',
  'CATMORF': 'Nome Própio Pessoa',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Gerúndio',
  'JUSTIFICATIVA': 'Gerúndio de almoçar, sem leitura convencional de nome de pessoa; confirmar que '
                   'não seja nomeação no corpus.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 316,
  'PALAVRA': 'alívio',
  'LEMA': 'alison',
  'CATMORF': 'Nome Própio Pessoa',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Singular',
  'JUSTIFICATIVA': 'Na leitura lexical, é nome comum masculino singular. Se a ocorrência for '
                   'apelido ou nome de personagem, manter a categoria atual.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 331,
  'PALAVRA': 'amarra',
  'LEMA': 'amareloo',
  'CATMORF': 'Nome Própio Pessoa',
  'CATEGORIA_SUGERIDA': 'Nome Comun Feminino Singular',
  'JUSTIFICATIVA': 'Na leitura lexical, é nome comum feminino singular. Se a ocorrência for '
                   'apelido ou nome de personagem, manter a categoria atual.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 344,
  'PALAVRA': 'amassar',
  'LEMA': 'amassadinha',
  'CATMORF': 'Nome Própio Pessoa',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Infinitivo',
  'JUSTIFICATIVA': 'Admite leitura de infinitivo; matá-la inclui pronome enclítico. Alimentar '
                   'também pode ser adjetivo. A proposta não estabelece preferência entre análises '
                   'possíveis.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 387,
  'PALAVRA': 'andou',
  'LEMA': 'andax',
  'CATMORF': 'Nome Própio Pessoa',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Indicativo Passado Terceira Pessoa Singular',
  'JUSTIFICATIVA': 'Pretérito perfeito de andar, terceira pessoa do singular.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 391,
  'PALAVRA': 'anel',
  'LEMA': 'andreia',
  'CATMORF': 'Nome Própio Pessoa',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Singular',
  'JUSTIFICATIVA': 'Na leitura lexical, é nome comum masculino singular. Se a ocorrência for '
                   'apelido ou nome de personagem, manter a categoria atual.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 392,
  'PALAVRA': 'anestesia',
  'LEMA': 'andréia',
  'CATMORF': 'Nome Própio Pessoa',
  'CATEGORIA_SUGERIDA': 'Nome Comun Feminino Singular',
  'JUSTIFICATIVA': 'Na leitura lexical, é nome comum feminino singular. Se a ocorrência for '
                   'apelido ou nome de personagem, manter a categoria atual.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 394,
  'PALAVRA': 'animada',
  'LEMA': 'ãnhan',
  'CATMORF': 'Nome Própio Pessoa',
  'CATEGORIA_SUGERIDA': 'Adjetivo Qualificativo Feminino Singular',
  'JUSTIFICATIVA': 'Admite leitura adjetival feminina ou de particípio; manter a categoria atual '
                   'se for apelido/personagem.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 395,
  'PALAVRA': 'animado',
  'LEMA': 'anhanh',
  'CATMORF': 'Nome Própio Pessoa',
  'CATEGORIA_SUGERIDA': 'Adjetivo Qualificativo Masculino Singular',
  'JUSTIFICATIVA': 'Admite leitura adjetival masculina ou de particípio; manter a categoria atual '
                   'se for apelido/personagem.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 408,
  'PALAVRA': 'antecipadamente',
  'LEMA': 'ant',
  'CATMORF': 'Nome Própio Pessoa',
  'CATEGORIA_SUGERIDA': 'Advérbio Geral',
  'JUSTIFICATIVA': 'Advérbio que exprime modo ou antecipação.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 504,
  'PALAVRA': 'apoia',
  'LEMA': 'apoia',
  'CATMORF': 'Nome Própio Pessoa',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Indicativo Presente Terceira Pessoa Singular',
  'JUSTIFICATIVA': 'Forma de apoiar; também admite imperativo, sem preferência obrigatória entre '
                   'análises verbais.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 520,
  'PALAVRA': 'arco-íris',
  'LEMA': 'arco-íris',
  'CATMORF': 'Nome Própio Pessoa',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Singular',
  'JUSTIFICATIVA': 'Na leitura lexical, é nome comum masculino singular. Se a ocorrência for '
                   'apelido ou nome de personagem, manter a categoria atual.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 544,
  'PALAVRA': 'arranhos',
  'LEMA': 'arranhos',
  'CATMORF': 'Nome Própio Pessoa',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Plural',
  'JUSTIFICATIVA': 'Na leitura lexical, é substantivo masculino no plural. Manter a categoria '
                   'atual se houver uso como nome próprio.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 678,
  'PALAVRA': 'atriz',
  'LEMA': 'atriz',
  'CATMORF': 'Nome Própio Pessoa',
  'CATEGORIA_SUGERIDA': 'Nome Comun Feminino Singular',
  'JUSTIFICATIVA': 'Na leitura lexical, é nome comum feminino singular. Se a ocorrência for '
                   'apelido ou nome de personagem, manter a categoria atual.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 770,
  'PALAVRA': 'baiacu',
  'LEMA': 'baiacu',
  'CATMORF': 'Nome Própio Pessoa',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Singular',
  'JUSTIFICATIVA': 'Na leitura lexical, é nome comum masculino singular. Se a ocorrência for '
                   'apelido ou nome de personagem, manter a categoria atual.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 778,
  'PALAVRA': 'balancinho',
  'LEMA': 'balancinho',
  'CATMORF': 'Nome Própio Pessoa',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Singular',
  'JUSTIFICATIVA': 'Na leitura lexical, é nome comum masculino singular. Se a ocorrência for '
                   'apelido ou nome de personagem, manter a categoria atual.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 786,
  'PALAVRA': 'baldinho',
  'LEMA': 'baldinho',
  'CATMORF': 'Nome Própio Pessoa',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Singular',
  'JUSTIFICATIVA': 'Na leitura lexical, é nome comum masculino singular. Se a ocorrência for '
                   'apelido ou nome de personagem, manter a categoria atual.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 950,
  'PALAVRA': 'biquíni',
  'LEMA': 'biquíni',
  'CATMORF': 'Nome Própio Pessoa',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Singular',
  'JUSTIFICATIVA': 'Na leitura lexical, é nome comum masculino singular. Se a ocorrência for '
                   'apelido ou nome de personagem, manter a categoria atual.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 971,
  'PALAVRA': 'bobeira',
  'LEMA': 'bobeira',
  'CATMORF': 'Nome Própio Pessoa',
  'CATEGORIA_SUGERIDA': 'Nome Comun Feminino Singular',
  'JUSTIFICATIVA': 'Na leitura lexical, é nome comum feminino singular. Se a ocorrência for '
                   'apelido ou nome de personagem, manter a categoria atual.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 983,
  'PALAVRA': 'bocona',
  'LEMA': 'bocona',
  'CATMORF': 'Nome Própio Pessoa',
  'CATEGORIA_SUGERIDA': 'Nome Comun Feminino Singular',
  'JUSTIFICATIVA': 'Na leitura lexical, é nome comum feminino singular. Se a ocorrência for '
                   'apelido ou nome de personagem, manter a categoria atual.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 995,
  'PALAVRA': 'boliche',
  'LEMA': 'boliche',
  'CATMORF': 'Nome Própio Pessoa',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Singular',
  'JUSTIFICATIVA': 'Na leitura lexical, é nome comum masculino singular. Se a ocorrência for '
                   'apelido ou nome de personagem, manter a categoria atual.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 1106,
  'PALAVRA': 'brinquedoteca',
  'LEMA': 'brinquedoteca',
  'CATMORF': 'Nome Própio Pessoa',
  'CATEGORIA_SUGERIDA': 'Nome Comun Feminino Singular',
  'JUSTIFICATIVA': 'Na leitura lexical, é nome comum feminino singular. Se a ocorrência for '
                   'apelido ou nome de personagem, manter a categoria atual.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 1121,
  'PALAVRA': 'brócolis',
  'LEMA': 'brócolis',
  'CATMORF': 'Nome Própio Pessoa',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Plural',
  'JUSTIFICATIVA': 'O alimento admite classificação como nome masculino plural; manter a categoria '
                   'atual se for apelido.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 1167,
  'PALAVRA': 'cabeçona',
  'LEMA': 'cabeçona',
  'CATMORF': 'Nome Própio Pessoa',
  'CATEGORIA_SUGERIDA': 'Nome Comun Feminino Singular',
  'JUSTIFICATIVA': 'Na leitura lexical, é nome comum feminino singular. Se a ocorrência for '
                   'apelido ou nome de personagem, manter a categoria atual.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 1346,
  'PALAVRA': 'carona',
  'LEMA': 'carona',
  'CATMORF': 'Nome Própio Pessoa',
  'CATEGORIA_SUGERIDA': 'Nome Comun Feminino Singular',
  'JUSTIFICATIVA': 'Na leitura lexical, é nome comum feminino singular. Se a ocorrência for '
                   'apelido ou nome de personagem, manter a categoria atual.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 1589,
  'PALAVRA': 'chuveirinho',
  'LEMA': 'chuveirinho',
  'CATMORF': 'Nome Própio Pessoa',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Singular',
  'JUSTIFICATIVA': 'Na leitura lexical, é nome comum masculino singular. Se a ocorrência for '
                   'apelido ou nome de personagem, manter a categoria atual.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 1591,
  'PALAVRA': 'chuveirão',
  'LEMA': 'chuveirão',
  'CATMORF': 'Nome Própio Pessoa',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Singular',
  'JUSTIFICATIVA': 'Na leitura lexical, é nome comum masculino singular. Se a ocorrência for '
                   'apelido ou nome de personagem, manter a categoria atual.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 1653,
  'PALAVRA': 'cochilinho',
  'LEMA': 'cochilinho',
  'CATMORF': 'Nome Própio Pessoa',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Singular',
  'JUSTIFICATIVA': 'Na leitura lexical, é nome comum masculino singular. Se a ocorrência for '
                   'apelido ou nome de personagem, manter a categoria atual.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 1664,
  'PALAVRA': 'cofrinho',
  'LEMA': 'cofrinho',
  'CATMORF': 'Nome Própio Pessoa',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Singular',
  'JUSTIFICATIVA': 'Na leitura lexical, é nome comum masculino singular. Se a ocorrência for '
                   'apelido ou nome de personagem, manter a categoria atual.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 1932,
  'PALAVRA': 'contos',
  'LEMA': 'contos',
  'CATMORF': 'Nome Própio Pessoa',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Plural',
  'JUSTIFICATIVA': 'Na leitura lexical, é substantivo masculino no plural. Manter a categoria '
                   'atual se houver uso como nome próprio.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 1965,
  'PALAVRA': 'coquetel',
  'LEMA': 'coquetel',
  'CATMORF': 'Nome Própio Pessoa',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Singular',
  'JUSTIFICATIVA': 'Na leitura lexical, é nome comum masculino singular. Se a ocorrência for '
                   'apelido ou nome de personagem, manter a categoria atual.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 2041,
  'PALAVRA': 'crepom',
  'LEMA': 'crepom',
  'CATMORF': 'Nome Própio Pessoa',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Singular',
  'JUSTIFICATIVA': 'Na leitura lexical, é nome comum masculino singular. Se a ocorrência for '
                   'apelido ou nome de personagem, manter a categoria atual.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 2063,
  'PALAVRA': 'crocodila',
  'LEMA': 'crocodila',
  'CATMORF': 'Nome Própio Pessoa',
  'CATEGORIA_SUGERIDA': 'Nome Comun Feminino Singular',
  'JUSTIFICATIVA': 'Na leitura lexical, é nome comum feminino singular. Se a ocorrência for '
                   'apelido ou nome de personagem, manter a categoria atual.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 2481,
  'PALAVRA': 'dinossaura',
  'LEMA': 'dinossaura',
  'CATMORF': 'Nome Própio Pessoa',
  'CATEGORIA_SUGERIDA': 'Nome Comun Feminino Singular',
  'JUSTIFICATIVA': 'Na leitura lexical, é nome comum feminino singular. Se a ocorrência for '
                   'apelido ou nome de personagem, manter a categoria atual.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 2636,
  'PALAVRA': 'dureza',
  'LEMA': 'dureza',
  'CATMORF': 'Nome Própio Pessoa',
  'CATEGORIA_SUGERIDA': 'Nome Comun Feminino Singular',
  'JUSTIFICATIVA': 'Na leitura lexical, é nome comum feminino singular. Se a ocorrência for '
                   'apelido ou nome de personagem, manter a categoria atual.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 3281,
  'PALAVRA': 'facona',
  'LEMA': 'facona',
  'CATMORF': 'Nome Própio Pessoa',
  'CATEGORIA_SUGERIDA': 'Nome Comun Feminino Singular',
  'JUSTIFICATIVA': 'Na leitura lexical, é nome comum feminino singular. Se a ocorrência for '
                   'apelido ou nome de personagem, manter a categoria atual.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 3349,
  'PALAVRA': 'fazendinha',
  'LEMA': 'fazendinha',
  'CATMORF': 'Nome Própio Pessoa',
  'CATEGORIA_SUGERIDA': 'Nome Comun Feminino Singular',
  'JUSTIFICATIVA': 'Na leitura lexical, é nome comum feminino singular. Se a ocorrência for '
                   'apelido ou nome de personagem, manter a categoria atual.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 3457,
  'PALAVRA': 'filhona',
  'LEMA': 'filhona',
  'CATMORF': 'Nome Própio Pessoa',
  'CATEGORIA_SUGERIDA': 'Nome Comun Feminino Singular',
  'JUSTIFICATIVA': 'Na leitura lexical, é nome comum feminino singular. Se a ocorrência for '
                   'apelido ou nome de personagem, manter a categoria atual.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 3466,
  'PALAVRA': 'filmadora',
  'LEMA': 'filmadora',
  'CATMORF': 'Nome Própio Pessoa',
  'CATEGORIA_SUGERIDA': 'Nome Comun Feminino Singular',
  'JUSTIFICATIVA': 'Na leitura lexical, é nome comum feminino singular. Se a ocorrência for '
                   'apelido ou nome de personagem, manter a categoria atual.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 3747,
  'PALAVRA': 'geleca',
  'LEMA': 'geleca',
  'CATMORF': 'Nome Própio Pessoa',
  'CATEGORIA_SUGERIDA': 'Nome Comun Feminino Singular',
  'JUSTIFICATIVA': 'Na leitura lexical, é nome comum feminino singular. Se a ocorrência for '
                   'apelido ou nome de personagem, manter a categoria atual.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 3749,
  'PALAVRA': 'geleia',
  'LEMA': 'geleia',
  'CATMORF': 'Nome Própio Pessoa',
  'CATEGORIA_SUGERIDA': 'Nome Comun Feminino Singular',
  'JUSTIFICATIVA': 'Na leitura lexical, é nome comum feminino singular. Se a ocorrência for '
                   'apelido ou nome de personagem, manter a categoria atual.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 3789,
  'PALAVRA': 'gol',
  'LEMA': 'gol',
  'CATMORF': 'Nome Própio Pessoa',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Singular',
  'JUSTIFICATIVA': 'Na leitura lexical, é nome comum masculino singular. Se a ocorrência for '
                   'apelido ou nome de personagem, manter a categoria atual.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 3794,
  'PALAVRA': 'gols',
  'LEMA': 'gols',
  'CATMORF': 'Nome Própio Pessoa',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Plural',
  'JUSTIFICATIVA': 'Na leitura lexical, é substantivo masculino no plural. Manter a categoria '
                   'atual se houver uso como nome próprio.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 3886,
  'PALAVRA': 'guarda-sol',
  'LEMA': 'guarda-sol',
  'CATMORF': 'Nome Própio Pessoa',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Singular',
  'JUSTIFICATIVA': 'Na leitura lexical, é nome comum masculino singular. Se a ocorrência for '
                   'apelido ou nome de personagem, manter a categoria atual.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 3921,
  'PALAVRA': 'hamster',
  'LEMA': 'hamster',
  'CATMORF': 'Nome Própio Pessoa',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Singular',
  'JUSTIFICATIVA': 'Na leitura lexical, é nome comum masculino singular. Se a ocorrência for '
                   'apelido ou nome de personagem, manter a categoria atual.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 4168,
  'PALAVRA': 'jabuticabeira',
  'LEMA': 'jabuticabeira',
  'CATMORF': 'Nome Própio Pessoa',
  'CATEGORIA_SUGERIDA': 'Nome Comun Feminino Singular',
  'JUSTIFICATIVA': 'Na leitura lexical, é nome comum feminino singular. Se a ocorrência for '
                   'apelido ou nome de personagem, manter a categoria atual.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 4303,
  'PALAVRA': 'lancinho',
  'LEMA': 'lancinho',
  'CATMORF': 'Nome Própio Pessoa',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Singular',
  'JUSTIFICATIVA': 'Na leitura lexical, é nome comum masculino singular. Se a ocorrência for '
                   'apelido ou nome de personagem, manter a categoria atual.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 4570,
  'PALAVRA': 'magos',
  'LEMA': 'magos',
  'CATMORF': 'Nome Própio Pessoa',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Plural',
  'JUSTIFICATIVA': 'Na leitura lexical, é substantivo masculino no plural. Manter a categoria '
                   'atual se houver uso como nome próprio.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 4677,
  'PALAVRA': 'maria-fumaça',
  'LEMA': 'maria-fumaça',
  'CATMORF': 'Nome Própio Pessoa',
  'CATEGORIA_SUGERIDA': 'Nome Comun Feminino Singular',
  'JUSTIFICATIVA': 'Na leitura lexical, é nome comum feminino singular. Se a ocorrência for '
                   'apelido ou nome de personagem, manter a categoria atual.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 4686,
  'PALAVRA': 'marionetes',
  'LEMA': 'marionetes',
  'CATMORF': 'Nome Própio Pessoa',
  'CATEGORIA_SUGERIDA': 'Nome Comun Feminino Plural',
  'JUSTIFICATIVA': 'Na leitura lexical, são substantivos femininos plurais; manter se nomearem '
                   'personagem ou pessoa.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 4687,
  'PALAVRA': 'maritaca',
  'LEMA': 'maritaca',
  'CATMORF': 'Nome Própio Pessoa',
  'CATEGORIA_SUGERIDA': 'Nome Comun Feminino Singular',
  'JUSTIFICATIVA': 'Na leitura lexical, é nome comum feminino singular. Se a ocorrência for '
                   'apelido ou nome de personagem, manter a categoria atual.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 4725,
  'PALAVRA': 'matá-la',
  'LEMA': 'matá-la',
  'CATMORF': 'Nome Própio Pessoa',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Infinitivo',
  'JUSTIFICATIVA': 'Admite leitura de infinitivo; matá-la inclui pronome enclítico. Alimentar '
                   'também pode ser adjetivo. A proposta não estabelece preferência entre análises '
                   'possíveis.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 4766,
  'PALAVRA': 'menta',
  'LEMA': 'menta',
  'CATMORF': 'Nome Própio Pessoa',
  'CATEGORIA_SUGERIDA': 'Nome Comun Feminino Singular',
  'JUSTIFICATIVA': 'Na leitura lexical, é nome comum feminino singular. Se a ocorrência for '
                   'apelido ou nome de personagem, manter a categoria atual.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 4800,
  'PALAVRA': 'metrô',
  'LEMA': 'metrô',
  'CATMORF': 'Nome Própio Pessoa',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Singular',
  'JUSTIFICATIVA': 'Na leitura lexical, é nome comum masculino singular. Se a ocorrência for '
                   'apelido ou nome de personagem, manter a categoria atual.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 4888,
  'PALAVRA': 'moletom',
  'LEMA': 'moletom',
  'CATMORF': 'Nome Própio Pessoa',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Singular',
  'JUSTIFICATIVA': 'Na leitura lexical, é nome comum masculino singular. Se a ocorrência for '
                   'apelido ou nome de personagem, manter a categoria atual.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 5226,
  'PALAVRA': 'objeto',
  'LEMA': 'objeto',
  'CATMORF': 'Nome Própio Pessoa',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Singular',
  'JUSTIFICATIVA': 'Na leitura lexical, é nome comum masculino singular. Se a ocorrência for '
                   'apelido ou nome de personagem, manter a categoria atual.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 5250,
  'PALAVRA': 'olhadinha',
  'LEMA': 'olhadinha',
  'CATMORF': 'Nome Própio Pessoa',
  'CATEGORIA_SUGERIDA': 'Nome Comun Feminino Singular',
  'JUSTIFICATIVA': 'Na leitura lexical, é nome comum feminino singular. Se a ocorrência for '
                   'apelido ou nome de personagem, manter a categoria atual.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 5296,
  'PALAVRA': 'orelhona',
  'LEMA': 'orelhona',
  'CATMORF': 'Nome Própio Pessoa',
  'CATEGORIA_SUGERIDA': 'Nome Comun Feminino Singular',
  'JUSTIFICATIVA': 'Na leitura lexical, é nome comum feminino singular. Se a ocorrência for '
                   'apelido ou nome de personagem, manter a categoria atual.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 5384,
  'PALAVRA': 'panelona',
  'LEMA': 'panelona',
  'CATMORF': 'Nome Própio Pessoa',
  'CATEGORIA_SUGERIDA': 'Nome Comun Feminino Singular',
  'JUSTIFICATIVA': 'Na leitura lexical, é nome comum feminino singular. Se a ocorrência for '
                   'apelido ou nome de personagem, manter a categoria atual.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 5385,
  'PALAVRA': 'panetone',
  'LEMA': 'panetone',
  'CATMORF': 'Nome Própio Pessoa',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Singular',
  'JUSTIFICATIVA': 'Na leitura lexical, é nome comum masculino singular. Se a ocorrência for '
                   'apelido ou nome de personagem, manter a categoria atual.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 5388,
  'PALAVRA': 'papagaia',
  'LEMA': 'papagaia',
  'CATMORF': 'Nome Própio Pessoa',
  'CATEGORIA_SUGERIDA': 'Nome Comun Feminino Singular',
  'JUSTIFICATIVA': 'Na leitura lexical, é nome comum feminino singular. Se a ocorrência for '
                   'apelido ou nome de personagem, manter a categoria atual.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 5407,
  'PALAVRA': 'papoula',
  'LEMA': 'papoula',
  'CATMORF': 'Nome Própio Pessoa',
  'CATEGORIA_SUGERIDA': 'Nome Comun Feminino Singular',
  'JUSTIFICATIVA': 'Na leitura lexical, é nome comum feminino singular. Se a ocorrência for '
                   'apelido ou nome de personagem, manter a categoria atual.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 5684,
  'PALAVRA': 'petecona',
  'LEMA': 'petecona',
  'CATMORF': 'Nome Própio Pessoa',
  'CATEGORIA_SUGERIDA': 'Nome Comun Feminino Singular',
  'JUSTIFICATIVA': 'Na leitura lexical, é nome comum feminino singular. Se a ocorrência for '
                   'apelido ou nome de personagem, manter a categoria atual.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 5866,
  'PALAVRA': 'porco-espinho',
  'LEMA': 'porco-espinho',
  'CATMORF': 'Nome Própio Pessoa',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Singular',
  'JUSTIFICATIVA': 'Na leitura lexical, é nome comum masculino singular. Se a ocorrência for '
                   'apelido ou nome de personagem, manter a categoria atual.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 5897,
  'PALAVRA': 'potinho',
  'LEMA': 'potinho',
  'CATMORF': 'Nome Própio Pessoa',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Singular',
  'JUSTIFICATIVA': 'Na leitura lexical, é nome comum masculino singular. Se a ocorrência for '
                   'apelido ou nome de personagem, manter a categoria atual.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 6102,
  'PALAVRA': 'puladeira',
  'LEMA': 'puladeira',
  'CATMORF': 'Nome Própio Pessoa',
  'CATEGORIA_SUGERIDA': 'Nome Comun Feminino Singular',
  'JUSTIFICATIVA': 'Na leitura lexical, é nome comum feminino singular. Se a ocorrência for '
                   'apelido ou nome de personagem, manter a categoria atual.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 6336,
  'PALAVRA': 'recreio',
  'LEMA': 'recreio',
  'CATMORF': 'Nome Própio Pessoa',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Singular',
  'JUSTIFICATIVA': 'Na leitura lexical, é nome comum masculino singular. Se a ocorrência for '
                   'apelido ou nome de personagem, manter a categoria atual.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 6456,
  'PALAVRA': 'Rio de Janeiro',
  'LEMA': 'Rio de Janeiro',
  'CATMORF': 'Nome Própio Pessoa',
  'CATEGORIA_SUGERIDA': 'Nome Própio Lugar',
  'JUSTIFICATIVA': 'Nome geográfico; corrigir para Lugar se a ocorrência designar a cidade ou o '
                   'estado.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 6877,
  'PALAVRA': 'sobrancelhas',
  'LEMA': 'sobrancelhas',
  'CATMORF': 'Nome Própio Pessoa',
  'CATEGORIA_SUGERIDA': 'Nome Comun Feminino Plural',
  'JUSTIFICATIVA': 'Na leitura lexical, são substantivos femininos plurais; manter se nomearem '
                   'personagem ou pessoa.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 6964,
  'PALAVRA': 'superbola',
  'LEMA': 'superbola',
  'CATMORF': 'Nome Própio Pessoa',
  'CATEGORIA_SUGERIDA': 'Nome Comun Feminino Singular',
  'JUSTIFICATIVA': 'Na leitura lexical, é nome comum feminino singular. Se a ocorrência for '
                   'apelido ou nome de personagem, manter a categoria atual.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 7054,
  'PALAVRA': 'tartarugona',
  'LEMA': 'tartarugona',
  'CATMORF': 'Nome Própio Pessoa',
  'CATEGORIA_SUGERIDA': 'Nome Comun Feminino Singular',
  'JUSTIFICATIVA': 'Na leitura lexical, é nome comum feminino singular. Se a ocorrência for '
                   'apelido ou nome de personagem, manter a categoria atual.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 7310,
  'PALAVRA': 'trapalhada',
  'LEMA': 'trapalhada',
  'CATMORF': 'Nome Própio Pessoa',
  'CATEGORIA_SUGERIDA': 'Nome Comun Feminino Singular',
  'JUSTIFICATIVA': 'Na leitura lexical, é nome comum feminino singular. Se a ocorrência for '
                   'apelido ou nome de personagem, manter a categoria atual.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 7637,
  'PALAVRA': 'viradinha',
  'LEMA': 'viradinha',
  'CATMORF': 'Nome Própio Pessoa',
  'CATEGORIA_SUGERIDA': 'Nome Comun Feminino Singular',
  'JUSTIFICATIVA': 'Na leitura lexical, é nome comum feminino singular. Se a ocorrência for '
                   'apelido ou nome de personagem, manter a categoria atual.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 7680,
  'PALAVRA': 'vixi',
  'LEMA': 'vixi',
  'CATMORF': 'Nome Própio Pessoa',
  'CATEGORIA_SUGERIDA': 'Interjeição',
  'JUSTIFICATIVA': 'Expressão exclamativa, com alongamento gráfico em alguns casos. Preservar a '
                   'grafia; confirmar que não seja nomeação.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 7732,
  'PALAVRA': 'xiii',
  'LEMA': 'xiii',
  'CATMORF': 'Nome Própio Pessoa',
  'CATEGORIA_SUGERIDA': 'Interjeição',
  'JUSTIFICATIVA': 'Expressão exclamativa, com alongamento gráfico em alguns casos. Preservar a '
                   'grafia; confirmar que não seja nomeação.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 7733,
  'PALAVRA': 'xiiii',
  'LEMA': 'xiiii',
  'CATMORF': 'Nome Própio Pessoa',
  'CATEGORIA_SUGERIDA': 'Interjeição',
  'JUSTIFICATIVA': 'Expressão exclamativa, com alongamento gráfico em alguns casos. Preservar a '
                   'grafia; confirmar que não seja nomeação.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 7734,
  'PALAVRA': 'xiiiii',
  'LEMA': 'xiiiii',
  'CATMORF': 'Nome Própio Pessoa',
  'CATEGORIA_SUGERIDA': 'Interjeição',
  'JUSTIFICATIVA': 'Expressão exclamativa, com alongamento gráfico em alguns casos. Preservar a '
                   'grafia; confirmar que não seja nomeação.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 7735,
  'PALAVRA': 'xiiiiii',
  'LEMA': 'xiiiiii',
  'CATMORF': 'Nome Própio Pessoa',
  'CATEGORIA_SUGERIDA': 'Interjeição',
  'JUSTIFICATIVA': 'Expressão exclamativa, com alongamento gráfico em alguns casos. Preservar a '
                   'grafia; confirmar que não seja nomeação.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 7764,
  'PALAVRA': 'zíper',
  'LEMA': 'zíper',
  'CATMORF': 'Nome Própio Pessoa',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Singular',
  'JUSTIFICATIVA': 'Na leitura lexical, é nome comum masculino singular. Se a ocorrência for '
                   'apelido ou nome de personagem, manter a categoria atual.',
  'DECISAO': 'Aprovada'}]
# Aplicação efetiva na base em memória:
alteracoes_grupo = aplicar_correcoes_categoria(resultado, correcoes_pessoas)
alteracoes.extend(alteracoes_grupo)
correcoes_aprovadas.extend(correcoes_pessoas)
print("Grupo: Nome Própio Pessoa")
print("Correções aprovadas:", len(correcoes_pessoas))
print("Aplicadas nesta execução:", len(alteracoes_grupo))
print("Já estavam corrigidas:", len(correcoes_pessoas) - len(alteracoes_grupo))
display(pd.DataFrame(correcoes_pessoas)[["PALAVRA", "LEMA", "CATMORF", "CATEGORIA_SUGERIDA", "DECISAO"]])


Grupo: Nome Própio Pessoa
Correções aprovadas: 82
Aplicadas nesta execução: 0
Já estavam corrigidas: 82


PALAVRA,LEMA,CATMORF,CATEGORIA_SUGERIDA,DECISAO
abdômen,abdômen,Nome Própio Pessoa,Nome Comun Masculino Singular,Aprovada
ajudei,ajudaro,Nome Própio Pessoa,Verbo Principal Indicativo Passado Primeira Pessoa Singular,Aprovada
alimentar,alice,Nome Própio Pessoa,Verbo Principal Infinitivo,Aprovada
almoçando,all,Nome Própio Pessoa,Verbo Principal Gerúndio,Aprovada
alívio,alison,Nome Própio Pessoa,Nome Comun Masculino Singular,Aprovada
amarra,amareloo,Nome Própio Pessoa,Nome Comun Feminino Singular,Aprovada
amassar,amassadinha,Nome Própio Pessoa,Verbo Principal Infinitivo,Aprovada
andou,andax,Nome Própio Pessoa,Verbo Principal Indicativo Passado Terceira Pessoa Singular,Aprovada
anel,andreia,Nome Própio Pessoa,Nome Comun Masculino Singular,Aprovada
anestesia,andréia,Nome Própio Pessoa,Nome Comun Feminino Singular,Aprovada


## 1.4. Verbo Principal Infinitivo — aplicar 11 correções aprovadas


In [6]:
# Lista completa das decisões aprovadas — Verbo Principal Infinitivo
correcoes_infinitivo = [{'LINHA_EXCEL': 234,
  'PALAVRA': 'ajuda',
  'LEMA': 'ajeitar',
  'CATMORF': 'Verbo Principal Infinitivo',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Indicativo Presente Terceira Pessoa Singular',
  'JUSTIFICATIVA': 'Forma de ajudar no presente do indicativo; também admite imperativo e '
                   'substantivo. A análise proposta é possível, sem preferência obrigatória. Não é '
                   'infinitivo na forma escrita e está registrada como paroxítona.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 240,
  'PALAVRA': 'ajudaram',
  'LEMA': 'ajudar',
  'CATMORF': 'Verbo Principal Infinitivo',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Indicativo Passado Terceira Pessoa Plural',
  'JUSTIFICATIVA': 'Pela palavra escrita, é forma de ajudar na terceira pessoa do plural do '
                   'passado, não infinitivo. A acentuação cadastrada como oxítona sugere '
                   'divergência com a escrita; esta proposta segue PALAVRA, sem alterar '
                   'transcrição ou acentuação.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 248,
  'PALAVRA': 'aladin',
  'LEMA': 'alar',
  'CATMORF': 'Verbo Principal Infinitivo',
  'CATEGORIA_SUGERIDA': 'Nome Própio Pessoa',
  'JUSTIFICATIVA': 'Admite leitura como nome do personagem Aladin/Aladim; a forma escrita não '
                   'corresponde a infinitivo português. Preservar a grafia e o lema.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 257,
  'PALAVRA': 'alcançou',
  'LEMA': 'alcançar',
  'CATMORF': 'Verbo Principal Infinitivo',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Indicativo Passado Terceira Pessoa Singular',
  'JUSTIFICATIVA': 'Pretérito perfeito de alcançar, terceira pessoa do singular; não é a forma '
                   'infinitiva alcançar.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 285,
  'PALAVRA': 'aline',
  'LEMA': 'alimentar',
  'CATMORF': 'Verbo Principal Infinitivo',
  'CATEGORIA_SUGERIDA': 'Nome Própio Pessoa',
  'JUSTIFICATIVA': 'Admite leitura como nome pessoal Aline; a forma escrita não corresponde ao '
                   'infinitivo alimentar indicado em LEMA. Não alterar o lema nesta etapa.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 290,
  'PALAVRA': 'almofada',
  'LEMA': 'almoçar',
  'CATMORF': 'Verbo Principal Infinitivo',
  'CATEGORIA_SUGERIDA': 'Nome Comun Feminino Singular',
  'JUSTIFICATIVA': 'Admite substantivo feminino singular; também é possível forma de almofadar. '
                   'Não corresponde ao infinitivo almoçar indicado em LEMA.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 328,
  'PALAVRA': 'amarelinha',
  'LEMA': 'amar',
  'CATMORF': 'Verbo Principal Infinitivo',
  'CATEGORIA_SUGERIDA': 'Nome Comun Feminino Singular',
  'JUSTIFICATIVA': 'Admite substantivo feminino singular, como o jogo amarelinha; também há '
                   'leitura adjetival. A proposta é uma análise possível, sem preferência entre as '
                   'classes válidas.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 337,
  'PALAVRA': 'amarro',
  'LEMA': 'amarrar',
  'CATMORF': 'Verbo Principal Infinitivo',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Indicativo Presente Primeira Pessoa Singular',
  'JUSTIFICATIVA': 'Presente do indicativo de amarrar, primeira pessoa do singular; não é '
                   'infinitivo.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 347,
  'PALAVRA': 'amassou',
  'LEMA': 'amassar',
  'CATMORF': 'Verbo Principal Infinitivo',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Indicativo Passado Terceira Pessoa Singular',
  'JUSTIFICATIVA': 'Pretérito perfeito de amassar, terceira pessoa do singular; não é infinitivo.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 384,
  'PALAVRA': 'andava',
  'LEMA': 'andar',
  'CATMORF': 'Verbo Principal Infinitivo',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Indicativo Imperfeito Terceira Pessoa Singular',
  'JUSTIFICATIVA': 'Pretérito imperfeito de andar. Primeira e terceira pessoas do singular são '
                   'possíveis; propõe-se terceira pessoa, sem preferência obrigatória.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 407,
  'PALAVRA': 'anta',
  'LEMA': 'anotar',
  'CATMORF': 'Verbo Principal Infinitivo',
  'CATEGORIA_SUGERIDA': 'Nome Comun Feminino Singular',
  'JUSTIFICATIVA': 'Admite substantivo feminino singular, nome do animal; não corresponde ao '
                   'infinitivo anotar indicado em LEMA.',
  'DECISAO': 'Aprovada'}]
# Aplicação efetiva na base em memória:
alteracoes_grupo = aplicar_correcoes_categoria(resultado, correcoes_infinitivo)
alteracoes.extend(alteracoes_grupo)
correcoes_aprovadas.extend(correcoes_infinitivo)
print("Grupo: Verbo Principal Infinitivo")
print("Correções aprovadas:", len(correcoes_infinitivo))
print("Aplicadas nesta execução:", len(alteracoes_grupo))
print("Já estavam corrigidas:", len(correcoes_infinitivo) - len(alteracoes_grupo))
display(pd.DataFrame(correcoes_infinitivo)[["PALAVRA", "LEMA", "CATMORF", "CATEGORIA_SUGERIDA", "DECISAO"]])


Grupo: Verbo Principal Infinitivo
Correções aprovadas: 11
Aplicadas nesta execução: 0
Já estavam corrigidas: 11


PALAVRA,LEMA,CATMORF,CATEGORIA_SUGERIDA,DECISAO
ajuda,ajeitar,Verbo Principal Infinitivo,Verbo Principal Indicativo Presente Terceira Pessoa Singular,Aprovada
ajudaram,ajudar,Verbo Principal Infinitivo,Verbo Principal Indicativo Passado Terceira Pessoa Plural,Aprovada
aladin,alar,Verbo Principal Infinitivo,Nome Própio Pessoa,Aprovada
alcançou,alcançar,Verbo Principal Infinitivo,Verbo Principal Indicativo Passado Terceira Pessoa Singular,Aprovada
aline,alimentar,Verbo Principal Infinitivo,Nome Própio Pessoa,Aprovada
almofada,almoçar,Verbo Principal Infinitivo,Nome Comun Feminino Singular,Aprovada
amarelinha,amar,Verbo Principal Infinitivo,Nome Comun Feminino Singular,Aprovada
amarro,amarrar,Verbo Principal Infinitivo,Verbo Principal Indicativo Presente Primeira Pessoa Singular,Aprovada
amassou,amassar,Verbo Principal Infinitivo,Verbo Principal Indicativo Passado Terceira Pessoa Singular,Aprovada
andava,andar,Verbo Principal Infinitivo,Verbo Principal Indicativo Imperfeito Terceira Pessoa Singular,Aprovada


## 1.5. Verbo Principal Indicativo Presente Terceira Pessoa Singular — aplicar 28 correções aprovadas


In [17]:
# Lista completa das decisões aprovadas — Verbo Principal Indicativo Presente Terceira Pessoa Singular
correcoes_presente_terceira = [{'LINHA_EXCEL': 225,
  'PALAVRA': 'aiaii',
  'LEMA': 'aiaiaiai',
  'CATMORF': 'Verbo Principal Indicativo Presente Terceira Pessoa Singular',
  'CATEGORIA_SUGERIDA': 'Interjeição',
  'JUSTIFICATIVA': 'Representação expressiva de ai repetido, compatível com o lema aiaiaiai; '
                   'propor interjeição, preservando a grafia.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 231,
  'PALAVRA': 'ajeitando',
  'LEMA': 'ajeitar',
  'CATMORF': 'Verbo Principal Indicativo Presente Terceira Pessoa Singular',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Gerúndio',
  'JUSTIFICATIVA': 'Gerúndio de ajeitar, não presente do indicativo na terceira pessoa.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 251,
  'PALAVRA': 'alaranjados',
  'LEMA': 'alaranjar',
  'CATMORF': 'Verbo Principal Indicativo Presente Terceira Pessoa Singular',
  'CATEGORIA_SUGERIDA': 'Adjetivo Qualificativo Masculino Plural',
  'JUSTIFICATIVA': 'Forma masculina plural de alaranjado; também admite análise de particípio '
                   'plural. Nenhuma dessas leituras corresponde a presente na terceira pessoa do '
                   'singular.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 255,
  'PALAVRA': 'alcançar',
  'LEMA': 'alçar',
  'CATMORF': 'Verbo Principal Indicativo Presente Terceira Pessoa Singular',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Infinitivo',
  'JUSTIFICATIVA': 'São formas de infinitivo. Leituras nominais também são possíveis, sem '
                   'preferência obrigatória; não são presente na terceira pessoa do singular.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 256,
  'PALAVRA': 'alcanço',
  'LEMA': 'alcançar',
  'CATMORF': 'Verbo Principal Indicativo Presente Terceira Pessoa Singular',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Indicativo Presente Primeira Pessoa Singular',
  'JUSTIFICATIVA': 'Na leitura verbal, é primeira pessoa do singular de alcançar; a terceira '
                   'pessoa é alcança. Também admite substantivo, sem preferência obrigatória.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 295,
  'PALAVRA': 'almoço',
  'LEMA': 'almoçar',
  'CATMORF': 'Verbo Principal Indicativo Presente Terceira Pessoa Singular',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Indicativo Presente Primeira Pessoa Singular',
  'JUSTIFICATIVA': 'Na leitura verbal, é primeira pessoa do singular de almoçar; a terceira pessoa '
                   'é almoça. Também admite substantivo masculino.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 302,
  'PALAVRA': 'alto',
  'LEMA': 'alterar',
  'CATMORF': 'Verbo Principal Indicativo Presente Terceira Pessoa Singular',
  'CATEGORIA_SUGERIDA': 'Adjetivo Qualificativo Masculino Singular',
  'JUSTIFICATIVA': 'Admite adjetivo masculino singular; também pode ser substantivo ou advérbio. '
                   'Não corresponde à terceira pessoa de alterar indicada no lema.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 320,
  'PALAVRA': 'amado',
  'LEMA': 'amar',
  'CATMORF': 'Verbo Principal Indicativo Presente Terceira Pessoa Singular',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Particípio Singular Masculino',
  'JUSTIFICATIVA': 'Particípio masculino singular de amar; também admite adjetivo ou nome. A '
                   'terceira pessoa do presente é ama.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 342,
  'PALAVRA': 'amassado',
  'LEMA': 'amassar',
  'CATMORF': 'Verbo Principal Indicativo Presente Terceira Pessoa Singular',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Particípio Singular Masculino',
  'JUSTIFICATIVA': 'Particípio masculino singular de amassar; também admite adjetivo ou '
                   'substantivo. A terceira pessoa do presente é amassa.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 355,
  'PALAVRA': 'amendoim',
  'LEMA': 'ameliezinha',
  'CATMORF': 'Verbo Principal Indicativo Presente Terceira Pessoa Singular',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Singular',
  'JUSTIFICATIVA': 'Substantivo masculino singular que designa a planta ou a semente; não é forma '
                   'verbal na categoria atual.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 373,
  'PALAVRA': 'amém',
  'LEMA': 'amém',
  'CATMORF': 'Verbo Principal Indicativo Presente Terceira Pessoa Singular',
  'CATEGORIA_SUGERIDA': 'Interjeição',
  'JUSTIFICATIVA': 'Expressão de concordância ou encerramento de oração; também admite '
                   'substantivo. Não é terceira pessoa do presente do indicativo.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 381,
  'PALAVRA': 'andar',
  'LEMA': 'andar',
  'CATMORF': 'Verbo Principal Indicativo Presente Terceira Pessoa Singular',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Infinitivo',
  'JUSTIFICATIVA': 'São formas de infinitivo. Leituras nominais também são possíveis, sem '
                   'preferência obrigatória; não são presente na terceira pessoa do singular.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 399,
  'PALAVRA': 'aniversario',
  'LEMA': 'aninhar',
  'CATMORF': 'Verbo Principal Indicativo Presente Terceira Pessoa Singular',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Singular',
  'JUSTIFICATIVA': 'Admite leitura como aniversário, preservando a ausência do acento. Sem acento, '
                   'também pode ser primeira pessoa de aniversariar; a terceira pessoa desse verbo '
                   'é aniversaria. A proposta nominal é uma análise possível.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 409,
  'PALAVRA': 'antecipado',
  'LEMA': 'antar',
  'CATMORF': 'Verbo Principal Indicativo Presente Terceira Pessoa Singular',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Particípio Singular Masculino',
  'JUSTIFICATIVA': 'Particípio masculino singular de antecipar; também admite adjetivo. Não é '
                   'terceira pessoa do presente.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 1204,
  'PALAVRA': 'cadê',
  'LEMA': 'cadê',
  'CATMORF': 'Verbo Principal Indicativo Presente Terceira Pessoa Singular',
  'CATEGORIA_SUGERIDA': 'Advérbio Geral',
  'JUSTIFICATIVA': 'Expressão interrogativa equivalente a onde está/onde estão, tradicionalmente '
                   'classificada como advérbio interrogativo. Advérbio Geral é o rótulo compatível '
                   'disponível na base, sem criar classe nova.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 1606,
  'PALAVRA': 'cineminha',
  'LEMA': 'cineminha',
  'CATMORF': 'Verbo Principal Indicativo Presente Terceira Pessoa Singular',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Singular',
  'JUSTIFICATIVA': 'Diminutivo nominal de cinema, masculino singular.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 2965,
  'PALAVRA': 'esconde-esconde',
  'LEMA': 'esconde-esconde',
  'CATMORF': 'Verbo Principal Indicativo Presente Terceira Pessoa Singular',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Singular',
  'JUSTIFICATIVA': 'Nome da brincadeira; o composto completo não é forma verbal simples do '
                   'presente.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 3273,
  'PALAVRA': 'fabricia',
  'LEMA': 'fabricia',
  'CATMORF': 'Verbo Principal Indicativo Presente Terceira Pessoa Singular',
  'CATEGORIA_SUGERIDA': 'Nome Própio Pessoa',
  'JUSTIFICATIVA': 'Admite leitura como nome de pessoa ou personagem (Fabrícia, Luíza, Mortícia), '
                   'preservando a grafia cadastrada. Não alterar o lema.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 4506,
  'PALAVRA': 'luiza',
  'LEMA': 'luiza',
  'CATMORF': 'Verbo Principal Indicativo Presente Terceira Pessoa Singular',
  'CATEGORIA_SUGERIDA': 'Nome Própio Pessoa',
  'JUSTIFICATIVA': 'Admite leitura como nome de pessoa ou personagem (Fabrícia, Luíza, Mortícia), '
                   'preservando a grafia cadastrada. Não alterar o lema.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 4742,
  'PALAVRA': 'meia-calça',
  'LEMA': 'meia-calça',
  'CATMORF': 'Verbo Principal Indicativo Presente Terceira Pessoa Singular',
  'CATEGORIA_SUGERIDA': 'Nome Comun Feminino Singular',
  'JUSTIFICATIVA': 'Nome feminino singular da peça de vestuário; o composto não corresponde à '
                   'forma verbal indicada.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 4756,
  'PALAVRA': 'meme',
  'LEMA': 'meme',
  'CATMORF': 'Verbo Principal Indicativo Presente Terceira Pessoa Singular',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Singular',
  'JUSTIFICATIVA': 'Admite substantivo masculino singular, sem leitura convencional como terceira '
                   'pessoa do presente na forma cadastrada.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 4962,
  'PALAVRA': 'morticia',
  'LEMA': 'morticia',
  'CATMORF': 'Verbo Principal Indicativo Presente Terceira Pessoa Singular',
  'CATEGORIA_SUGERIDA': 'Nome Própio Pessoa',
  'JUSTIFICATIVA': 'Admite leitura como nome de pessoa ou personagem (Fabrícia, Luíza, Mortícia), '
                   'preservando a grafia cadastrada. Não alterar o lema.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 5869,
  'PALAVRA': 'porta-mala',
  'LEMA': 'porta-mala',
  'CATMORF': 'Verbo Principal Indicativo Presente Terceira Pessoa Singular',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Singular',
  'JUSTIFICATIVA': 'Admite leitura nominal de porta-malas, preservando a grafia singular do dado; '
                   'não classificar o composto como verbo simples.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 5903,
  'PALAVRA': 'poxa',
  'LEMA': 'poxa',
  'CATMORF': 'Verbo Principal Indicativo Presente Terceira Pessoa Singular',
  'CATEGORIA_SUGERIDA': 'Interjeição',
  'JUSTIFICATIVA': 'Expressão exclamativa de surpresa, contrariedade ou admiração.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 6102,
  'PALAVRA': 'pula-pula',
  'LEMA': 'pula-pula',
  'CATMORF': 'Verbo Principal Indicativo Presente Terceira Pessoa Singular',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Singular',
  'JUSTIFICATIVA': 'Nome masculino do brinquedo/aparelho ou da ave; a composição não equivale à '
                   'forma verbal simples pula.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 6455,
  'PALAVRA': 'riponga',
  'LEMA': 'riponga',
  'CATMORF': 'Verbo Principal Indicativo Presente Terceira Pessoa Singular',
  'CATEGORIA_SUGERIDA': 'Adjetivo Qualificativo Comum Singular',
  'JUSTIFICATIVA': 'Adjetivo de dois gêneros, também substantivo de dois gêneros, ligado a hippie. '
                   'A análise adjetival é uma opção válida, sem preferência obrigatória.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 6769,
  'PALAVRA': 'seresta',
  'LEMA': 'seresta',
  'CATMORF': 'Verbo Principal Indicativo Presente Terceira Pessoa Singular',
  'CATEGORIA_SUGERIDA': 'Nome Comun Feminino Singular',
  'JUSTIFICATIVA': 'Substantivo feminino singular para serenata ou manifestação musical; essa é a '
                   'leitura lexical proposta. Se houver emprego verbal neológico no corpus, manter '
                   'a categoria atual.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 7014,
  'PALAVRA': 'tamanduá',
  'LEMA': 'tamanduá',
  'CATMORF': 'Verbo Principal Indicativo Presente Terceira Pessoa Singular',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Singular',
  'JUSTIFICATIVA': 'Substantivo masculino singular que designa o animal.',
  'DECISAO': 'Aprovada'}]
# Aplicação efetiva na base em memória:
alteracoes_grupo = aplicar_correcoes_categoria(resultado, correcoes_presente_terceira)
alteracoes.extend(alteracoes_grupo)
correcoes_aprovadas.extend(correcoes_presente_terceira)
print("Grupo: Verbo Principal Indicativo Presente Terceira Pessoa Singular")
print("Correções aprovadas:", len(correcoes_presente_terceira))
print("Aplicadas nesta execução:", len(alteracoes_grupo))
print("Já estavam corrigidas:", len(correcoes_presente_terceira) - len(alteracoes_grupo))
display(pd.DataFrame(correcoes_presente_terceira)[["PALAVRA", "LEMA", "CATMORF", "CATEGORIA_SUGERIDA", "DECISAO"]])


Grupo: Verbo Principal Indicativo Presente Terceira Pessoa Singular
Correções aprovadas: 28
Aplicadas nesta execução: 0
Já estavam corrigidas: 28


,PALAVRA,LEMA,CATMORF,CATEGORIA_SUGERIDA,DECISAO
0,aiaii,aiaiaiai,Verbo Principal Indicativo Presente Terceira P...,Interjeição,Aprovada
1,ajeitando,ajeitar,Verbo Principal Indicativo Presente Terceira P...,Verbo Principal Gerúndio,Aprovada
2,alaranjados,alaranjar,Verbo Principal Indicativo Presente Terceira P...,Adjetivo Qualificativo Masculino Plural,Aprovada
3,alcançar,alçar,Verbo Principal Indicativo Presente Terceira P...,Verbo Principal Infinitivo,Aprovada
4,alcanço,alcançar,Verbo Principal Indicativo Presente Terceira P...,Verbo Principal Indicativo Presente Primeira P...,Aprovada
5,almoço,almoçar,Verbo Principal Indicativo Presente Terceira P...,Verbo Principal Indicativo Presente Primeira P...,Aprovada
6,alto,alterar,Verbo Principal Indicativo Presente Terceira P...,Adjetivo Qualificativo Masculino Singular,Aprovada
7,amado,amar,Verbo Principal Indicativo Presente Terceira P...,Verbo Principal Particípio Singular Masculino,Aprovada
8,amassado,amassar,Verbo Principal Indicativo Presente Terceira P...,Verbo Principal Particípio Singular Masculino,Aprovada
9,amendoim,ameliezinha,Verbo Principal Indicativo Presente Terceira P...,Nome Comun Masculino Singular,Aprovada


## 2. Palavras identificadas como fora do escopo

In [18]:
palavras_remover = ['atamedama', 'bibolo', 'cafeta', 'casma', 'cehgar', 'dexo', 'encontralo', 'esto', 'euxo', 'letre', 'moder', 'niclodaplo', 'nicrodoplo', 'ocabelo', 'opapaiurso', 'sarlso', 'siamem', 'noís', 'autie', 'bebeça', 'bibibiva', 'chemana', 'chocola', 'dade', 'dede', 'emelie', 'iolha', 'levantese', 'manamana', 'melie', 'migatina', 'mimie', 'moite', 'nacidade', 'nose', 'olele', 'parchiva', 'picala', 'puxavida', 'sentese', 'vendese', 'suvaco', 'luzona', 'mamadeirona', 'velocipede']
total_antes = len(resultado)
mascara_remocao = resultado["PALAVRA"].isin(palavras_remover)
removidas = resultado.loc[mascara_remocao].copy()
nao_encontradas = sorted(set(palavras_remover) - set(removidas["PALAVRA"]))
resultado = resultado.loc[~mascara_remocao].reset_index(drop=True)
quantidade_removida = len(removidas)
assert total_antes - quantidade_removida == len(resultado)
print("Remoções previstas nesta execução:", quantidade_removida)
print("Total previsto na base:", len(resultado))
if nao_encontradas:
    print("Palavras já removidas ou não encontradas:", nao_encontradas)
display(removidas[["PALAVRA", "LEMA", "CATMORF"]])


Remoções previstas nesta execução: 0
Total previsto na base: 7785
Palavras já removidas ou não encontradas: ['atamedama', 'autie', 'bebeça', 'bibibiva', 'bibolo', 'cafeta', 'casma', 'cehgar', 'chemana', 'chocola', 'dade', 'dede', 'dexo', 'emelie', 'encontralo', 'esto', 'euxo', 'iolha', 'letre', 'levantese', 'luzona', 'mamadeirona', 'manamana', 'melie', 'migatina', 'mimie', 'moder', 'moite', 'nacidade', 'niclodaplo', 'nicrodoplo', 'nose', 'noís', 'ocabelo', 'olele', 'opapaiurso', 'parchiva', 'picala', 'puxavida', 'sarlso', 'sentese', 'siamem', 'suvaco', 'velocipede', 'vendese']


,PALAVRA,LEMA,CATMORF


## 3. Salvamento e contagens

In [19]:
assert hashlib.sha256(arquivo.read_bytes()).hexdigest() == hash_entrada, "A fonte mudou durante a execução."
retidas = base_original.loc[~base_original["PALAVRA"].isin(palavras_remover)].reset_index(drop=True)
pd.testing.assert_frame_equal(retidas.drop(columns="CATMORF"), resultado.drop(columns="CATMORF"))
from tempfile import TemporaryDirectory
# A limpeza inclui arquivos de inspeção gerados pela ferramenta, mesmo em caso de erro.
with TemporaryDirectory(prefix="temporarios_", dir=arquivo.resolve().parent) as pasta_temporaria:
    temporario = Path(pasta_temporaria) / "base_verificacao.xlsx"
    pacote = Path(pasta_temporaria) / "operacao.json"
    matriz = [list(resultado.columns)] + json.loads(resultado.to_json(orient="values", force_ascii=False))
    pacote.write_text(json.dumps({
        "source": str(arquivo.resolve()), "target": str(temporario.resolve()),
        "originalRows": len(base_original) + 1, "values": matriz
    }, ensure_ascii=False), encoding="utf-8")
    runtime = Path(os.environ.get("USERPROFILE", str(Path.home()))) / ".cache/codex-runtimes/codex-primary-runtime/dependencies/node"
    node = runtime / "bin/node.exe"
    ambiente = os.environ.copy()
    ambiente["CODEX_NODE_PACKAGES"] = str(runtime / "node_modules")
    codigo_js = "\nimport fs from 'node:fs/promises';\nimport {createRequire} from 'node:module';\nimport {pathToFileURL} from 'node:url';\nconst require = createRequire(process.env.CODEX_NODE_PACKAGES + '/../package.json');\nconst {FileBlob, SpreadsheetFile} = await import(pathToFileURL(require.resolve('@oai/artifact-tool')).href);\nconst job=JSON.parse(await fs.readFile(process.argv[1],'utf8'));\nconst wb=await SpreadsheetFile.importXlsx(await FileBlob.load(job.source));\nconst sheet=wb.worksheets.getItem('Sheet1');\nsheet.getRange('A1:H'+job.originalRows).clear({applyTo:'contents'});\nsheet.getRange('A1:H'+job.values.length).values=job.values;\nwb.recalculate();\nconsole.log((await wb.inspect({kind:'region',sheetId:'Sheet1',range:'A1:C5',maxChars:1000})).ndjson);\nconst out=await SpreadsheetFile.exportXlsx(wb);\nawait out.save(job.target);\n"
    processo = subprocess.run([str(node), "--input-type=module", "-e", codigo_js, str(pacote.resolve())],
                             env=ambiente, capture_output=True, text=True, encoding="utf-8")
    if processo.returncode:
        raise RuntimeError(processo.stderr or processo.stdout)
    verificada = pd.read_excel(temporario)
    pd.testing.assert_frame_equal(resultado, verificada, check_dtype=False)
    assert hashlib.sha256(arquivo.read_bytes()).hexdigest() == hash_entrada
    if not resultado.equals(base_original):
        os.replace(temporario, arquivo)
print("Correções aplicadas nesta execução:", len(alteracoes))
print("Remoções nesta execução:", quantidade_removida)
print("Total atual na base:", len(verificada))
print("Lemas e demais colunas das linhas mantidas: preservados.")


Correções aplicadas nesta execução: 0
Remoções nesta execução: 0
Total atual na base: 7785
Lemas e demais colunas das linhas mantidas: preservados.
